In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AQ
Grok API Key not set (and this is optional)
Groq API Key exists and begins gsk_
OpenRouter API Key not set (and this is optional)


In [3]:
# Connect to client libraries

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

ollama = OpenAI(api_key="ollama", base_url=ollama_url)
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)



In [4]:
models = ["gpt-5", "llama3.2", "claude-sonnet-4-5-20250929", "grok-4", "gemini-3.5-flash-lite", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]

clients = {"gpt-5": openai, "llama3.2": ollama, "claude-sonnet-4-5-20250929": anthropic, "grok-4": grok, "gemini-3.5-flash-lite": gemini, "openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [11]:
example_response = """
{
  "schedule": [
    {
      "day": "Day 1",
      "title": "Full Body A",
      "blocks": [
        {
          "block": "Warm-up",
          "exercises": [
            {
              "name": "Incline Walk",
              "sets": "1",
              "reps": "5 min",
              "rest": "-",
              "notes": "Easy pace"
            }
          ]
        },
        {
          "block": "Main Strength",
          "exercises": [
            {
              "name": "Back Squat",
              "sets": "3",
              "reps": "5",
              "rest": "2–3 min",
              "notes": "RPE 6–7; tempo 2-1-1"
            }
          ]
        }
      ]
    }
  ]
}
"""

In [12]:
system_prompt = f"""
Your task is generate workout and schedule based from user preferences and goals.
Respond only with this format {example_response}. Do not provide any explanation other than occasional comments.
The response need to be efficient workout based user goals.
"""

def user_prompt_for(user_data):
    return f"""
Make workout schedule with the efficient way based from user preference and goals.
User data:
{user_data}
Response only with this format {example_response}.
"""

In [6]:
def message_for(user_data):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "system", "content": user_prompt_for(user_data)}
    ]

In [13]:
import json
from workout_renderer import render_workout_table

def generate_workout(model, user_data):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=message_for(user_data), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content

    workout_data = json.loads(reply)
    return render_workout_table(workout_data)

In [17]:
def build_data_and_generate(model, age,
    gender,
    height,
    weight,
    fitness_level,
    goals,
    workout_days,
    workout_duration,
    equipment,
    preferred_workout,
    limitations,
    additional_preferences
    ):
    user_data = {
        "age": age,
        "gender": gender,
        "height_cm": height,
        "weight_kg": weight,
        "fitness_level": fitness_level,
        "goals": goals,
        "workout_days_per_week": workout_days,
        "workout_duration_minutes": workout_duration,
        "equipment": equipment,
        "preferred_workout": preferred_workout,
        "limitations": limitations,
        "additional_preferences": additional_preferences,
    }

    print("Berhasil build user_data:", user_data)

    return generate_workout(model, user_data)

In [18]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Your personal coach") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            gr.Markdown("## User Profile")

            with gr.Row():
                age = gr.Number(
                    label="Age",
                    minimum=10,
                    maximum=100,
                    value=23
                )

                gender = gr.Dropdown(
                    choices=["Male", "Female"],
                    label="Gender",
                    value="Male"
                )

            with gr.Row():
                height = gr.Number(
                    label="Height (cm)",
                    minimum=100,
                    maximum=250,
                    value=171
                )

                weight = gr.Number(
                    label="Weight (kg)",
                    minimum=30,
                    maximum=300,
                    value=73
                )

            fitness_level = gr.Dropdown(
                choices=[
                    "Beginner",
                    "Intermediate",
                    "Advanced"
                ],
                label="Fitness Level",
                value="Beginner"
            )

            goal = gr.CheckboxGroup(
                choices=[
                    "Lose Weight",
                    "Build Muscle",
                    "Increase Strength",
                    "Improve Endurance",
                    "Improve General Fitness"
                ],
                label="Goals"
            )

            with gr.Row():
                workout_days = gr.Slider(
                    minimum=1,
                    maximum=7,
                    step=1,
                    value=3,
                    label="Workout Days / Week"
                )

                workout_duration = gr.Slider(
                    minimum=15,
                    maximum=180,
                    step=5,
                    value=60,
                    label="Workout Duration (minutes)"
                )

            equipment = gr.CheckboxGroup(
                choices=[
                    "No Equipment",
                    "Dumbbell",
                    "Barbell",
                    "Resistance Band",
                    "Pull-up Bar",
                    "Gym Machine",
                    "Full Gym"
                ],
                label="Available Equipment"
            )

            preferred_workout = gr.CheckboxGroup(
                choices=[
                    "Strength Training",
                    "Running",
                    "Cycling",
                    "Walking",
                    "HIIT",
                    "Mobility",
                    "Stretching"
                ],
                label="Preferred Workout"
            )

            limitations = gr.TextArea(
                label="Injuries / Physical Limitations",
                placeholder="Example: Knee pain, lower back pain, none...",
                lines=2
            )

            additional_preferences = gr.TextArea(
                label="Additional Preferences",
                placeholder="Example: I prefer short workouts and don't like running...",
                lines=3
            )

        # =========================
        # GENERATED WORKOUT
        # =========================
        with gr.Column(scale=6):

            gr.Markdown("## Workout Schedule")

            generated = gr.HTML(
                value="""
                <div class="empty-workout">
                    Your personalized workout will appear here.
                </div>
                """
            )

    with gr.Row(elem_classes=["controls"]):
        model = gr.Dropdown(models, value=models[0], show_label=False)
        generate = gr.Button(f'Generate Workout Schedule')

    generate.click(fn=build_data_and_generate, inputs=[
        model, 
        age,
        gender,
        height,
        weight,
        fitness_level,
        goal,
        workout_days,
        workout_duration,
        equipment,
        preferred_workout,
        limitations,
        additional_preferences
    ], 
    outputs=[generated])

ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Berhasil build user_data: {'age': 23, 'gender': 'Male', 'height_cm': 171, 'weight_kg': 73, 'fitness_level': 'Intermediate', 'goals': ['Lose Weight', 'Build Muscle'], 'workout_days_per_week': 3, 'workout_duration_minutes': 60, 'equipment': ['Full Gym'], 'preferred_workout': ['Strength Training'], 'limitations': '', 'additional_preferences': ''}
